In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('/content/drive/MyDrive/UHIMWS/INMET_S_SC_A806_FLORIANOPOLIS_01-01-2025_A_30-09-2025.CSV',
                 encoding='latin-1',
                 sep=';',
                 skiprows=8,
                 decimal=',',
                 na_values=['', 'NULL', 'NaN'],
                 dayfirst=True)

print(f"Shape do dataframe: {df.shape}")
print("\nPrimeiras linhas dos dados:")
print(df.head())

Shape do dataframe: (6552, 20)

Primeiras linhas dos dados:
         Data  Hora UTC  PRECIPITAÇÃO TOTAL, HORÁRIO (mm)  \
0  2025/01/01  0000 UTC                               0.0   
1  2025/01/01  0100 UTC                               0.0   
2  2025/01/01  0200 UTC                               0.0   
3  2025/01/01  0300 UTC                               8.2   
4  2025/01/01  0400 UTC                               1.0   

   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  \
0                                             1013.3       
1                                             1013.6       
2                                             1014.1       
3                                             1013.6       
4                                             1012.6       

   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)  \
0                                           1013.3   
1                                           1013.6   
2                                           1014.3

In [4]:
df_temp = df.copy()

In [5]:
df_temp['Data'] = pd.to_datetime(df_temp['Data'], format='%Y/%m/%d')

In [6]:
colunas_temperatura = {
        'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'temp_horaria',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'temp_max_hora_ant',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'temp_min_hora_ant',
        'TEMPERATURA DO PONTO DE ORVALHO (°C)': 'temp_orvalho',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade_relativa'
    }

In [7]:
for col_antiga, col_nova in colunas_temperatura.items():
        if col_antiga in df_temp.columns:
            df_temp.rename(columns={col_antiga: col_nova}, inplace=True)

In [8]:
colunas_interesse = ['Data', 'Hora UTC', 'temp_horaria', 'temp_max_hora_ant',
                        'temp_min_hora_ant', 'temp_orvalho', 'umidade_relativa']

In [9]:
df_temp = df_temp[colunas_interesse]

In [10]:
df_temp = df_temp[(df_temp['temp_horaria'] >= -10) & (df_temp['temp_horaria'] <= 50)]
df_temp = df_temp[(df_temp['temp_orvalho'] >= -10) & (df_temp['temp_orvalho'] <= 40)]

In [11]:
df_temp = df_temp.sort_values(['Data', 'Hora UTC'])

In [12]:
df_temp['datetime'] = pd.to_datetime(
        df_temp['Data'].astype(str) + ' ' + df_temp['Hora UTC'].str.replace(' UTC', ''),
        format='%Y-%m-%d %H%M'
    )

In [13]:
print("Temperaturas de Florianópolis")

print(f"Temperatura máxima: {df_temp['temp_horaria'].max():.2f}°C")
print(f"Temperatura mínima: {df_temp['temp_horaria'].min():.2f}°C")
print(f"Temperatura média: {df_temp['temp_horaria'].mean():.2f}°C")
print(f"Desvio padrão: {df_temp['temp_horaria'].std():.2f}°C")

Temperaturas de Florianópolis
Temperatura máxima: 36.90°C
Temperatura mínima: 13.80°C
Temperatura média: 24.37°C
Desvio padrão: 3.44°C


In [16]:
data_especifica = '2025-01-08'
hora_inicio = '0400'
hora_fim = '0800'

df_filtrado = df_temp[(df_temp['Data'].dt.strftime('%Y-%m-%d') == data_especifica) &
                      (df_temp['Hora UTC'].str.replace(' UTC', '') >= hora_inicio) &
                      (df_temp['Hora UTC'].str.replace(' UTC', '') <= hora_fim)]

print(f"\nTemperaturas para {data_especifica} entre {hora_inicio} e {hora_fim} UTC:")
print(f"Temperatura máxima: {df_filtrado['temp_horaria'].max():.2f}°C")
print(f"Temperatura mínima: {df_filtrado['temp_horaria'].min():.2f}°C")
print(f"Temperatura média: {df_filtrado['temp_horaria'].mean():.2f}°C")
print(f"Desvio padrão: {df_filtrado['temp_horaria'].std():.2f}°C")


Temperaturas para 2025-01-08 entre 0400 e 0800 UTC:
Temperatura máxima: 22.40°C
Temperatura mínima: 20.50°C
Temperatura média: 21.44°C
Desvio padrão: 0.80°C
